# AdaBoost Algorithm

Companion notebook for the [AdaBoost Algorithm wiki page](https://ml-viz.vercel.app/wiki/adaboost-algorithm).

We implement AdaBoost from scratch using decision stumps, visualize how sample
weights evolve, and plot the ensemble error curve as rounds accumulate.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(42)

## 1 — One-round worked trace

10 points, uniform weights, stump misclassifies 2 of them.

$$\epsilon_1 = 0.2 \quad\Longrightarrow\quad \alpha_1 = \tfrac{1}{2}\ln\frac{0.8}{0.2} = 0.693$$

In [ ]:
n = 10
w = np.full(n, 1/n)

# +1 = correctly classified, -1 = wrong (y_i * h_b(x_i))
agreement = np.ones(n)
agreement[:2] = -1   # first 2 are wrong

epsilon = w[agreement < 0].sum()
alpha   = 0.5 * np.log((1 - epsilon) / (epsilon + 1e-12))

w_new = w * np.exp(-alpha * agreement)
w_new /= w_new.sum()

print(f"ε = {epsilon:.3f}")
print(f"α = {alpha:.4f}  (= 0.5·ln(4) = {0.5*np.log(4):.4f})")
print(f"Correct weight: {w_new[2]:.4f} each")
print(f"Wrong weight:   {w_new[0]:.4f} each")
print(f"Sum of wrong weights: {w_new[:2].sum():.3f}  (should ≈ 0.5)")

## 2 — Decision stump

A depth-1 decision tree: pick feature $j$, threshold $t$, polarity $s \in\{+1,-1\}$,
and predict $s$ if $x_j \geq t$, else $-s$. Choose $(j, t, s)$ to minimise
the weighted misclassification error.

In [ ]:
class DecisionStump:
    def fit(self, X, y, w):
        n, d = X.shape
        best = {'err': np.inf}
        for j in range(d):
            for thresh in np.unique(X[:, j]):
                for sign in [1, -1]:
                    pred = np.where(X[:, j] >= thresh, sign, -sign)
                    err  = np.dot(w, pred != y)
                    if err < best['err']:
                        best = {'err': err, 'j': j, 'thresh': thresh, 'sign': sign}
        self.__dict__.update(best)
        return self

    def predict(self, X):
        return np.where(X[:, self.j] >= self.thresh, self.sign, -self.sign)

print("DecisionStump defined.")

## 3 — Full AdaBoost implementation

In [ ]:
class AdaBoost:
    def __init__(self, n_rounds=50):
        self.n_rounds = n_rounds

    def fit(self, X, y):
        n = len(y)
        w = np.full(n, 1/n)
        self.stumps, self.alphas = [], []
        self.train_errors_ = []
        self.round_errors_ = []

        for _ in range(self.n_rounds):
            h = DecisionStump().fit(X, y, w)
            pred = h.predict(X)
            eps   = np.dot(w, pred != y) + 1e-12
            alpha = 0.5 * np.log((1 - eps) / eps)

            self.round_errors_.append(eps)
            w *= np.exp(-alpha * y * pred)
            w /= w.sum()

            self.stumps.append(h)
            self.alphas.append(alpha)
            self.train_errors_.append((self.predict(X) != y).mean())

        return self

    def predict(self, X):
        votes = sum(a * h.predict(X) for a, h in zip(self.alphas, self.stumps))
        return np.sign(votes)

## 4 — 1-D toy dataset and training

In [ ]:
# 1-D binary classification: two Gaussian blobs with some overlap
n_pos, n_neg = 60, 60
X_pos = np.random.normal(1.5, 1.0, (n_pos, 1))
X_neg = np.random.normal(-1.5, 1.0, (n_neg, 1))
X_train = np.vstack([X_pos, X_neg])
y_train = np.hstack([np.ones(n_pos), -np.ones(n_neg)])

ada = AdaBoost(n_rounds=60)
ada.fit(X_train, y_train)

print(f"Final training error: {ada.train_errors_[-1]:.3f}")
print(f"α values (first 5): {[f'{a:.3f}' for a in ada.alphas[:5]]}")

## 5 — Visualizing the ensemble

In [ ]:
xgrid = np.linspace(-5, 5, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# -- Panel 1: decision boundary after 1, 10, 60 rounds --
colors = {'1': '#6366f1', '10': '#20d9d2', '60': '#f97316'}
for B, col in zip([1, 10, 60], colors.values()):
    votes = sum(ada.alphas[i] * ada.stumps[i].predict(xgrid)
                for i in range(B))
    axes[0].plot(xgrid, np.sign(votes), color=col, lw=1.5, label=f'B={B}')

axes[0].scatter(X_train[y_train==1, 0], np.ones(n_pos)*1.15, s=15,
                color='#6366f1', alpha=0.4)
axes[0].scatter(X_train[y_train==-1, 0], np.ones(n_neg)*(-1.15), s=15,
                color='#ef4444', alpha=0.4)
axes[0].axhline(0, color='#555', lw=0.5)
axes[0].set_title('Decision boundary evolution')
axes[0].legend(fontsize=9)
axes[0].set_ylim(-1.5, 1.5)

# -- Panel 2: training error vs rounds --
axes[1].plot(range(1, len(ada.train_errors_)+1), ada.train_errors_,
             color='#6366f1')
axes[1].set_xlabel('Number of rounds B')
axes[1].set_ylabel('Training error')
axes[1].set_title('Training error vs. rounds')
axes[1].grid(True, alpha=0.3)

# -- Panel 3: α weights per round --
axes[2].bar(range(1, len(ada.alphas)+1), ada.alphas,
            color='#f97316', alpha=0.8)
axes[2].set_xlabel('Round b')
axes[2].set_ylabel('Model weight αb')
axes[2].set_title('α weights per round')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1 — weighted error verification

After AdaBoost's weight update in round $b$, the **current stump** has exactly
$\epsilon = 0.5$ weighted error on the **new** weight distribution.
Verify this identity numerically for the first 5 rounds.

In [ ]:
# TODO(you): re-run the first 5 rounds and check that after the weight update,
# re-evaluating the *same* stump gives epsilon = 0.5 on the new weights.

# w = np.full(len(y_train), 1/len(y_train))
# for b in range(5):
#     h = ada.stumps[b]
#     pred = h.predict(X_train)
#     eps_before = np.dot(w, pred != y_train)
#     alpha = ada.alphas[b]
#     w *= np.exp(-alpha * y_train * pred)
#     w /= w.sum()
#     eps_after = ???
#     print(f"Round {b+1}: ε_before={eps_before:.4f}, ε_after={eps_after:.4f}")

# assert all values ε_after ≈ 0.5

### Exercise 2 — α as a function of ε

Plot $\alpha(\epsilon) = \frac{1}{2}\ln\frac{1-\epsilon}{\epsilon}$ for
$\epsilon \in (0, 1)$. Mark the three key points: $\epsilon\to 0^+$,
$\epsilon=0.5$, $\epsilon\to 1^-$.

<details>
<summary>Solution</summary>

```python
eps = np.linspace(0.01, 0.99, 300)
alpha = 0.5 * np.log((1 - eps) / eps)

plt.figure(figsize=(7, 4))
plt.plot(eps, alpha, color='#6366f1')
plt.axhline(0, color='#555', lw=0.5)
plt.axvline(0.5, color='#f97316', linestyle='--', label='ε=0.5 → α=0 (ignored)')
plt.xlabel('Weighted error ε'); plt.ylabel('Model weight α')
plt.title('α(ε): model weight as function of error')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()
```
</details>